# Rellenado de Series Temporales (Participación de Mercado)
## Fases de Tratado
### Fase 0: Configuración General

El objetivo de este script es procesar la base de datos histórica de Participación de Mercado, construyendo un andamiaje temporal completo (Scaffold). Esto garantiza que cada subpartida tenga un registro continuo mes a mes en la línea de tiempo, rellenando los vacíos con nulos reales para evitar rupturas en las gráficas, y calculando los porcentajes de participación dinámicos para México y China.

Dependencias requeridas:
- `pandas (pd):` Manipulación de datos, operaciones vectorizadas y manejo de series de tiempo.
- `os` / `pathlib (Path):` Manejo seguro de rutas relativas e interacciones con el sistema de archivos.

Variables Globales:
- **Rutas:** Directorios de entrada (Raw) y salida (Intermediate), así como los nombres de los archivos correspondientes.

In [1]:
import pandas as pd
import os
from pathlib import Path

# --- RUTAS DE ARCHIVOS ---
DIR_ENTRADA = "../data/raw"
ARCHIVO_ENTRADA = "participacion_total_hs6_countries.xlsx"
PATH_INPUT_PART = Path(DIR_ENTRADA) / ARCHIVO_ENTRADA

DIR_SALIDA = "../data/intermediate"
ARCHIVO_SALIDA = "Participacion_HTS_Completo.xlsx"
PATH_OUTPUT_PART = Path(DIR_SALIDA) / ARCHIVO_SALIDA

print("--- CONFIGURACIÓN CARGADA ---")
print(f"Input:  {PATH_INPUT_PART}")
print(f"Output: {PATH_OUTPUT_PART}")

### Fase 0.5: Definición de Funciones

Se encapsula la lógica de transformación de datos para facilitar el mantenimiento y la lectura:

**1. Funciones Auxiliares (`_nombre`)**
- **`_construir_andamiaje`**: Identifica el rango de fechas global (min/max) del conjunto de datos y crea un producto cartesiano (MultiIndex) entre todas las subpartidas únicas y todos los meses posibles, generando el "esqueleto" temporal continuo.

**2. Funciones Principales (`nombre`)**
- **`rellenar_series_participacion`**: Orquestador maestro. Carga el archivo tolerando inconsistencias de formato (Excel vs CSV), asegura el tipo de dato de las fechas, genera el andamiaje, cruza las tablas (Left Join) y calcula las columnas derivadas (porcentajes) antes de ordenar el resultado.

In [2]:
# --- FUNCIONES AUXILIARES ---

def _construir_andamiaje(df, col_id, col_fecha):
    """
    Crea un DataFrame base (Scaffold) con la combinación ininterrumpida 
    de todas las subpartidas y todos los meses del rango detectado.
    """
    min_date = df[col_fecha].min()
    max_date = df[col_fecha].max()
    print(f"   Rango temporal detectado: {min_date.date()} a {max_date.date()}")
    
    # Frecuencia 'MS' asegura inicio de mes (Month Start)
    todas_las_fechas = pd.date_range(start=min_date, end=max_date, freq='MS')
    subpartidas_unicas = df[col_id].unique()
    
    # Producto cartesiano
    multi_index = pd.MultiIndex.from_product(
        [subpartidas_unicas, todas_las_fechas], 
        names=[col_id, col_fecha]
    )
    
    return pd.DataFrame(index=multi_index).reset_index()


# --- FUNCIONES PRINCIPALES ---

def rellenar_series_participacion(ruta_input):
    """
    Orquestador principal: Extrae, formatea fechas, genera andamiaje, 
    rellena huecos temporales y calcula Market Share.
    """
    print(f">> Leyendo archivo desde: {ruta_input}")
    try:
        df = pd.read_excel(ruta_input)
    except ValueError:
        df = pd.read_csv(ruta_input)
    except FileNotFoundError:
        print(f"❌ ERROR: No se encontró el archivo en {ruta_input}")
        print("Verifica que estés ejecutando el script desde la carpeta correcta.")
        return pd.DataFrame()

    # 1. Asegurar formato de fecha explícito
    df['Fecha'] = pd.to_datetime(df['Fecha'])

    # 2. Crear estructura base ininterrumpida
    df_andamiaje = _construir_andamiaje(df, col_id='Subpartida', col_fecha='Fecha')

    # 3. Cruce Left Join para poblar el andamiaje
    print("   Uniendo datos originales con el andamiaje completo...")
    df_merged = pd.merge(df_andamiaje, df, on=['Subpartida', 'Fecha'], how='left')

    # 4. Cálculos Derivados (Market Share)
    print("   Calculando porcentajes de participación...")
    # Las filas vacías (huecos temporales rellenados) mantendrán NaN por reglas de Pandas
    df_merged['Mexico (%)'] = df_merged['Mexico'] / df_merged['Total']
    df_merged['China (%)'] = df_merged['China'] / df_merged['Total']

    # 5. Ordenamiento final
    df_merged = df_merged.sort_values(by=['Subpartida', 'Fecha'])
    
    return df_merged

### Fase 1: Procesamiento y Cálculo de Participación
Se invoca al orquestador. Se leerá la base cruda, se transformarán las fechas, se expandirá la base para asegurar series continuas y se calcularán los porcentajes proporcionales contra el Total.

In [3]:
DF_PARTICIPACION_COMPLETO = rellenar_series_participacion(PATH_INPUT_PART)
print("\n>> ¡Proceso de andamiaje temporal y cálculo completado!")

### Fase 2: Exportación Final
El DataFrame expandido y calculado se guarda en la ruta del directorio intermedio, creando la estructura de carpetas de manera automática en caso de ser necesario.

In [4]:
if not DF_PARTICIPACION_COMPLETO.empty:
    print(f"Generando Excel: {PATH_OUTPUT_PART}...")
    try:
        # Crear carpeta de salida preventivamente
        os.makedirs(DIR_SALIDA, exist_ok=True)
        
        # Exportación
        DF_PARTICIPACION_COMPLETO.to_excel(PATH_OUTPUT_PART, index=False)
        
        print("¡ÉXITO! Archivo guardado correctamente con las series rellenadas.")
        print(f"Total de registros procesados: {len(DF_PARTICIPACION_COMPLETO)}")
        print("\nVista previa:")
        print(DF_PARTICIPACION_COMPLETO.head())
        
    except PermissionError:
        print(f"❌ ERROR CRÍTICO: Asegúrate de cerrar el archivo '{PATH_OUTPUT_PART}' antes de ejecutar.")
    except Exception as e:
        print(f"❌ ERROR AL EXPORTAR: {e}")
else:
    print("⚠️ ADVERTENCIA: No se generaron registros, archivo no exportado.")